In [1]:
import onnx
from onnx import numpy_helper

In [2]:
# Load the ONNX model
model_path = "../simple_with_activation_linear_model.onnx"
model = onnx.load(model_path)

# Check model validity
onnx.checker.check_model(model)
graph = model.graph

print("=" * 60)
print(f"Model: {graph.name}  |  IR version: {model.ir_version}")
print("=" * 60)

# --- Inputs & Outputs ---
in_shape: int = 0
out_shape: int = 0
print("\nInputs:")
for inp in graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"  {inp.name}: {shape}")
    in_shape = shape[1] if len(shape) > 1 else 1

print("\nOutputs:")
for out in graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"  {out.name}: {shape}")
    out_shape = shape[1] if len(shape) > 1 else 1

# --- Build a weights lookup for quick access ---
weights = {init.name: numpy_helper.to_array(init) for init in graph.initializer}

# --- Iterate over layers (nodes) ---
print(f"\nLayers ({len(graph.node)} total):\n")
layers = []
for i, node in enumerate(graph.node):
    print(f"[{i:02d}] {node.op_type:20s} | name: {node.name or '(unnamed)'}")
    print(f"      inputs : {list(node.input)}")
    print(f"      outputs: {list(node.output)}")

    # Print attributes (e.g. kernel_shape, strides, etc.)
    for attr in node.attribute:
        if attr.type == onnx.AttributeProto.INT:
            val = attr.i
        elif attr.type == onnx.AttributeProto.FLOAT:
            val = attr.f
        elif attr.type == onnx.AttributeProto.INTS:
            val = list(attr.ints)
        elif attr.type == onnx.AttributeProto.FLOATS:
            val = list(attr.floats)
        elif attr.type == onnx.AttributeProto.STRING:
            val = attr.s.decode()
        else:
            val = "(complex type)"
        print(f"      attr   : {attr.name} = {val}")

    # Print shapes of any learnable weights attached to this node
    for inp_name in node.input:
        if inp_name in weights:
            w = weights[inp_name]
            print(f"      weight : {inp_name} → shape {w.shape}, dtype {w.dtype}")

Model: main_graph  |  IR version: 6

Inputs:
  onnx::Gemm_0: [1, 10]

Outputs:
  16: [1, 5]

Layers (8 total):

[00] Gemm                 | name: /linear/Gemm
      inputs : ['onnx::Gemm_0', 'linear.weight', 'linear.bias']
      outputs: ['/linear/Gemm_output_0']
      attr   : alpha = 1.0
      attr   : beta = 1.0
      attr   : transB = 1
      weight : linear.weight → shape (10, 10), dtype float32
      weight : linear.bias → shape (10,), dtype float32
[01] Relu                 | name: /act_1_relu/Relu
      inputs : ['/linear/Gemm_output_0']
      outputs: ['/act_1_relu/Relu_output_0']
[02] Gemm                 | name: /linear2/Gemm
      inputs : ['/act_1_relu/Relu_output_0', 'linear2.weight', 'linear2.bias']
      outputs: ['/linear2/Gemm_output_0']
      attr   : alpha = 1.0
      attr   : beta = 1.0
      attr   : transB = 1
      weight : linear2.weight → shape (20, 10), dtype float32
      weight : linear2.bias → shape (20,), dtype float32
[03] Sigmoid              | name: /a